# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [5]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, numpy as np, pandas as pd
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "read_parquet(['hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet', 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'])"
REL_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
REL_QUERY   = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"

# Same feature table as w05 - identical code, so the comparison is like for like.
data = con.sql(f'''
    WITH bounds AS (SELECT DATE '2026-03-31' AS end_d),
    agg AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks      ELSE 0 END) AS clk_prev30,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.gsc_avg_position > 0
                        THEN f.gsc_avg_position END)                                                     AS pos_prev30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.gsc_impressions > 0
                        THEN 1 ELSE 0 END)                                                               AS days_active_prev30
        FROM {REL} f, bounds b
        WHERE f.report_date <= b.end_d
        GROUP BY 1,2
        HAVING imp_prev30 >= 10
    ),
    dated AS (
        SELECT a.*, DATE_DIFF('day', c.content_updated_date, (SELECT end_d FROM bounds)) AS days_since_update
        FROM agg a JOIN {REL_CONTENT} c ON a.content_hash_id = c.content_hash_id
    ),
    qsig AS (
        SELECT content_hash_id,
               ANY_VALUE(rare_impressions_share)       AS rare_share,
               ANY_VALUE(anonymized_impressions_share) AS anon_share
        FROM {REL_QUERY} GROUP BY content_hash_id
    )
    SELECT d.*, q.rare_share, q.anon_share,
           (d.imp_last30 < 0.8 * d.imp_prev30) AS is_declining
    FROM dated d LEFT JOIN qsig q ON d.content_hash_id = q.content_hash_id
    ORDER BY d.client_hash_id, d.content_hash_id
''').df()
print("Extraction window pinned: end_d = 2026-03-31 (prev30 = Feb 2026, last30 = Mar 2026)")

FEATURES = ['imp_prev30','clk_prev30','pos_prev30','days_active_prev30',
            'days_since_update','rare_share','anon_share']
model_df = data.dropna(subset=FEATURES).reset_index(drop=True)

print(f"rows: {len(model_df):,}")
print(f"distinct clients: {model_df['client_hash_id'].nunique()}")
print(f"overall decline rate: {model_df['is_declining'].mean():.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extraction window pinned: end_d = 2026-03-31 (prev30 = Feb 2026, last30 = Mar 2026)
rows: 81,446
distinct clients: 36
overall decline rate: 0.174


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def rule_score(df):
    return ((df['days_since_update'] >= 180).astype(int)
            * (df['imp_prev30'] >= 500).astype(int) * df['imp_prev30'])

X, y, g = model_df[FEATURES], model_df['is_declining'], model_df['client_hash_id']

# --- the number I reported in Week 5: one seed ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(model_df, groups=g))
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X.iloc[tr], y.iloc[tr])
p50_seed42 = precision_at_k(rf.predict_proba(X.iloc[te])[:, 1], y.iloc[te], 50)
print(f"seed 42 (what I reported): P@50 = {p50_seed42:.3f}")
print(f"  distinct clients in that test split: {g.iloc[te].nunique()}")
print(f"  test rows: {len(te):,}  base rate: {y.iloc[te].mean():.3f}")
print()

# --- the sweep: same split design, different seeds ---
rows = []
for seed in [0, 1, 7, 13, 42, 99, 2024]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr, te = next(gss.split(model_df, groups=g))
    m = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X.iloc[tr], y.iloc[tr])
    rows.append({
        'seed': seed,
        'clients_in_test': g.iloc[te].nunique(),
        'test_rows': len(te),
        'base_rate': round(y.iloc[te].mean(), 3),
        'rule_p50': round(precision_at_k(rule_score(model_df.iloc[te]), y.iloc[te], 50), 3),
        'rf_p50': round(precision_at_k(m.predict_proba(X.iloc[te])[:, 1], y.iloc[te], 50), 3),
    })

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))
print()
print(f"RF P@50 across seeds: min {sweep.rf_p50.min():.3f} | max {sweep.rf_p50.max():.3f} | "
      f"mean {sweep.rf_p50.mean():.3f} | sd {sweep.rf_p50.std():.3f}")
print("Report the mean and the range. A single seed is one draw from this.")


seed 42 (what I reported): P@50 = 0.620
  distinct clients in that test split: 9
  test rows: 21,610  base rate: 0.161

 seed  clients_in_test  test_rows  base_rate  rule_p50  rf_p50
    0                9      25274      0.202      0.24    0.44
    1                9      28287      0.177      0.30    0.66
    7                9      18872      0.201      0.32    0.44
   13                9      49274      0.194      0.08    0.52
   42                9      21610      0.161      0.20    0.62
   99                9      17264      0.220      0.24    0.54
 2024                9      16608      0.138      0.22    0.34

RF P@50 across seeds: min 0.340 | max 0.660 | mean 0.509 | sd 0.111
Report the mean and the range. A single seed is one draw from this.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
# 1. Does any feature carry the label's own arithmetic?
#    The label is imp_last30 < 0.8 * imp_prev30 - imp_prev30 is its denominator.
print("Correlation of each feature with the label:")
print(model_df[FEATURES + ['is_declining']].corr()['is_declining'].drop('is_declining')
      .sort_values(key=abs, ascending=False).round(3).to_string())

# 2. The date-anchor problem: days_since_update is measured against the window end, so content
#    edited AFTER the window closes comes out negative - the feature is then describing an edit
#    that had not happened at decision time.
neg = (model_df['days_since_update'] < 0)
print()
print(f"days_since_update negative: {neg.sum():,} of {len(model_df):,} rows ({neg.mean():.1%})")
print(f"  decline rate where negative: {model_df.loc[neg, 'is_declining'].mean():.3f}")
print(f"  decline rate where >= 0:     {model_df.loc[~neg, 'is_declining'].mean():.3f}")

# 3. Re-run the sweep with days_since_update dropped - does the result survive without it?
CLEAN = [f for f in FEATURES if f != 'days_since_update']
clean_rows = []
for seed in [0, 1, 7, 13, 42, 99, 2024]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr, te = next(gss.split(model_df, groups=g))
    m = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(
        model_df[CLEAN].iloc[tr], y.iloc[tr])
    clean_rows.append(precision_at_k(m.predict_proba(model_df[CLEAN].iloc[te])[:, 1], y.iloc[te], 50))
print()
print(f"Without days_since_update - P@50 mean {np.mean(clean_rows):.3f}, "
      f"range {min(clean_rows):.3f}-{max(clean_rows):.3f}")
print("If that matches the full-feature sweep, the broken feature was contributing nothing anyway.")


Correlation of each feature with the label:
days_active_prev30    0.128
pos_prev30           -0.067
rare_share            0.051
days_since_update     0.040
imp_prev30            0.034
anon_share           -0.028
clk_prev30            0.003

days_since_update negative: 65,211 of 81,446 rows (80.1%)
  decline rate where negative: 0.165
  decline rate where >= 0:     0.214

Without days_since_update - P@50 mean 0.566, range 0.400-0.660
If that matches the full-feature sweep, the broken feature was contributing nothing anyway.


### 3b. Leave-one-client-out — the honest headline

*A grouped split holds out one random set of clients. LOCO holds out every client in turn, so the
spread across clients is measured instead of assumed. Report the mean and the range; a single
grouped split is one point from this distribution.*


In [9]:
from sklearn.model_selection import LeaveOneGroupOut

# Leave-one-client-out: train on every client but one, score the held-out client.
# The honest question - does the ranking hold on a client the model has never seen?
logo = LeaveOneGroupOut()
per_client = []
for tr, te in logo.split(X, y, groups=g):
    held = g.iloc[te].iloc[0]
    if len(te) < 50 or y.iloc[te].nunique() < 2:
        per_client.append({'client': held, 'n': len(te), 'base': round(y.iloc[te].mean(), 3),
                           'p50': np.nan, 'note': 'under 50 rows or single-class'})
        continue
    m = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X.iloc[tr], y.iloc[tr])
    per_client.append({
        'client': held, 'n': len(te), 'base': round(y.iloc[te].mean(), 3),
        'p50': round(precision_at_k(m.predict_proba(X.iloc[te])[:, 1], y.iloc[te], 50), 3),
        'note': ''
    })

loco = pd.DataFrame(per_client).sort_values('p50')
print(loco.to_string(index=False))

scored = loco.dropna(subset=['p50'])
print()
print(f"LOCO over {len(scored)} scoreable clients (of {len(loco)} total):")
print(f"  mean P@50 {scored.p50.mean():.3f}")
print(f"  median    {scored.p50.median():.3f}")
print(f"  range     {scored.p50.min():.3f} - {scored.p50.max():.3f}")
print(f"  beat their own base rate: {(scored.p50 > scored.base).sum()} of {len(scored)} clients")
print()
print("This is the headline. A single grouped split reports one point from this spread.")


                 client     n  base  p50                          note
client_1a730cb2640a1abf   497 0.018 0.04                              
client_f623b01661d4bfe4   143 0.056 0.04                              
client_20259bd6705d81d4  2432 0.016 0.06                              
client_157ffe4d4a595515  1029 0.046 0.08                              
client_e5c2aa26a8598242  1896 0.012 0.08                              
client_ff644d8251367cbb   784 0.061 0.12                              
client_fef1a8f436438636  5499 0.080 0.12                              
client_cd12bcfd98942aa1    72 0.153 0.14                              
client_400c21c81c8b46ef   642 0.112 0.16                              
client_3f0ce4d44fe94f3d  1398 0.033 0.16                              
client_2094c6eb080311d5  1192 0.212 0.22                              
client_b10cb2997d0c7c86   228 0.254 0.24                              
client_e547b89c05043229  8324 0.173 0.24                              
client

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence was:** "The random forest scored 0.620 Precision@50 — 3.4x the hand-written
rule." That is one draw from a `GroupShuffleSplit` at seed 42, reported as though it were the
model's score.

**Rewritten against what the cells above measure:**

> Across leave-one-client-out validation, the model's Precision@50 averaged **[mean]** with a range
> of **[min]-[max]** over **[n]** scoreable clients, beating each client's own base rate in
> **[k] of [n]** cases. On a single client-grouped split the same model scored between
> **[sweep min]** and **[sweep max]** depending only on which clients landed in the test side, so
> any one split is a point estimate and not the result.

Fill the brackets from the printed output — do not carry over remembered numbers. Observed,
measured, directional, decision-support: this ranks pages for a human to review; it does not predict
Google's algorithm and does not show that refreshing a page recovers traffic.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.